# The WSP Heuristic

Here we calculate the WSPD for a given problem and then check whether the number of edges crossing the each pair of clusters is the correct number given the number of exits out of the cluster as per the proved theorems. 

## Using internal python implementation of WSPD class

In [1]:
import sys, importlib
from pathlib import Path

import tsplib95
import numpy as np

from wsp import ds, tsp
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

TREE_TYPE = ds.PKPRQuadTree
SIZE_THRESHOLD = 250
TSP_LIB_PATH = Path("TSPLIB")
S_FACTOR = 1.5
#ax = np.array([None, None])
#sys.setrecursionlimit(500_000)

In [2]:
#all_problems : dict[str, tsp.TravellingSalesmanProblem[TREE_TYPE]] = {}

#for file in sorted(TSP_LIB_PATH.iterdir()): # Loop through every tsp
#    if file.suffix != ".tsp" or not file.is_file():
#        continue
#    problem = tsplib95.load(f"{file.absolute()}")
#    if problem.edge_weight_type != "EUC_2D": # Skip non-Euclidean TSPs
#        continue
#    if problem.dimension > SIZE_THRESHOLD: # Skip large TSPs
#        continue

#    points = [ds.Point(*problem.node_coords[i]) for i in problem.get_nodes()]
#    ts_problem = tsp.TravellingSalesmanProblem[TREE_TYPE](TREE_TYPE, points, ax, s=S_FACTOR) 
    
#    all_problems[problem.name] = ts_problem
#    print(f"Added {problem.name}")

#print("Found", len(all_problems), "euclidean TSPs")

In [3]:
#for name, problem in all_problems.items():
#    tour, _, _ = problem.nnn_path
#    badCount = 0
#    for (A, B), _ in problem.wspd:
#        pointsA = set(A.covered_points)
#        pointsB = set(B.covered_points)
#        if not problem.wsp_heuristic_good(tour, pointsA, pointsB):
#            badCount += 1
#    if badCount > 0:
#        print(f"Problem {name} has {badCount} bad pairs")

## Faster Runner using external cpython libs
still solving problem as they are seen

no bad pairs found up to 12k for s=1.5  (TSPLIB EUC2D)
no bad pairs found up to 12k for s=1.0 (TSPLIB EUC2D)

In [4]:
%load_ext line_profiler
import itertools

import numba as nb
nb.config.THREADING_LAYER = 'tbb'
from numba import njit
from numba.typed import List as NumbaList
import elkai

from utils.sort_by_key_inplace import sort_by_key_inplace
from utils.wspd_euc import get_wspd

SIZE_THRESHOLD2 = 500

In [5]:
#all_problems : list[tsplib95.models.StandardProblem] = []

#for file in sorted(TSP_LIB_PATH.iterdir()): # Loop through every tsp
#    if file.suffix != ".tsp" or not file.is_file():
#        continue
#    problem = tsplib95.load(f"{file.absolute()}")
#    if problem.edge_weight_type != "EUC_2D": # Skip non-Euclidean TSPs
#        continue
#    if problem.dimension > SIZE_THRESHOLD2 or problem.dimension <= 350: # Skip large TSPs
#        continue
#    all_problems.append(problem)

#len(all_problems)

In [6]:
@njit(cache=True, inline="always", boundscheck=False, fastmath=True)
def wsp_heuristic_good(a_list: np.ndarray, b_list: np.ndarray, pos_in_tour: np.ndarray) -> bool:
    """A heuristic to check if a path is good based on the WSPs

    Takes in the positions of the nodes in A and B in the tour (sorted), and the total number of points in the problem.
    a_pos \subseteq [0, num_points-1] are the positions of the nodes in A in the tour
    b_pos \subseteq [0, num_points-1] are the positions of the nodes in B in the tour

    We then check the following conditions:
    1. If there are no exit pairs with endpoints in separate sets, then there must be exactly two edges connecting A and B
    2. If there are an even number of exit pairs with endpoints in separate sets, then there must be no edges connecting A and B 
    3. If there are an odd number of exit pairs with endpoints in separate sets, then there must be exactly one edge connecting A and B
    """
    num_points = len(pos_in_tour)
    #assert len(a_pos) > 0 and len(b_pos) > 0, "Sets must be non-empty"

    # keep track of how many edges directly connect A and B in the tour
    biconn_AB_count = 0
    biconn_BA_count = 0
    # keep track of how many exit pairs have endpoints in separate sets
    exit_AA_count = 0
    exit_BB_count = 0
    exit_AB_count = 0
    exit_BA_count = 0

    # a_pos[i] = pos_in_tour[a_list[i]]

    # check the edge cases
    if pos_in_tour[a_list[-1]] == num_points - 1 and pos_in_tour[b_list[0]] == 0:
        biconn_AB_count += 1
    elif pos_in_tour[b_list[-1]] == num_points - 1 and pos_in_tour[a_list[0]] == 0:
        biconn_BA_count += 1
    elif (pos_in_tour[a_list[-1]] == num_points - 1 and pos_in_tour[a_list[0]] == 0) or (pos_in_tour[b_list[-1]] == num_points - 1 and pos_in_tour[b_list[0]] == 0):
        pass # then there is just an a loop and nothing to see here
    elif pos_in_tour[a_list[-1]] > pos_in_tour[b_list[-1]]:
        if pos_in_tour[a_list[0]] < pos_in_tour[b_list[0]]:
            exit_AA_count += 1
        else:
            exit_AB_count += 1
    else:
        if pos_in_tour[b_list[0]] < pos_in_tour[a_list[0]]:
            exit_BB_count += 1
        else:
            exit_BA_count += 1

    # two pointer approach to count biconns and exits
    i = j = 0
    na = len(a_list)
    nb = len(b_list)

    # tracking info
    next_a = pos_in_tour[a_list[0]]
    next_b = pos_in_tour[b_list[0]]
    exited_A = None

    while True:
        if next_a <= next_b:
            # handle if exited
            if exited_A is not None:
                if exited_A:
                    exit_AA_count += 1
                else:
                    exit_BA_count += 1
                exited_A = None

            # check if biconn or exit
            if next_a + 1 == next_b:
                biconn_AB_count += 1
            elif i + 1 >= na or next_a + 1 != pos_in_tour[a_list[i+1]]:
                exited_A = True

            # increment i and update next_a
            i += 1
            if i < na:
                next_a = pos_in_tour[a_list[i]]
            else:
                break
        else:
            # handle if exited
            if exited_A is not None:
                if exited_A:
                    exit_AB_count += 1
                else:
                    exit_BB_count += 1
                exited_A = None

            # check if biconn or exit
            if next_b + 1 == next_a:
                biconn_BA_count += 1
            elif j + 1 >= nb or next_b + 1 != pos_in_tour[b_list[j+1]]:
                exited_A = False
            
            # increment j and update next_b
            j += 1
            if j < nb:
                next_b = pos_in_tour[b_list[j]]
            else:
                break
    # at this point one or both sets are expended
    while i < na:
        if exited_A is not None:
            if exited_A:
                exit_AA_count += 1
            else:
                exit_BA_count += 1
            exited_A = None
        if i + 1 >= na or pos_in_tour[a_list[i]] + 1 != pos_in_tour[a_list[i+1]]:
            exited_A = True
        i += 1
    while j < nb:
        if exited_A is not None:
            if exited_A:
                exit_AB_count += 1
            else:
                exit_BB_count += 1
            exited_A = None
        if j + 1 >= nb or pos_in_tour[b_list[j]] + 1 != pos_in_tour[b_list[j+1]]:
            exited_A = False
        j += 1
        

    ## covers both single in outs and multi case
    cross_exits = exit_AB_count + exit_BA_count
    if cross_exits == 0:
        return biconn_AB_count == 1 and biconn_BA_count == 1
    elif cross_exits % 2 == 0:
        return biconn_AB_count == 0 and biconn_BA_count == 0
    else:
        return (biconn_AB_count == 1 and biconn_BA_count == 0) or (biconn_AB_count == 0 and biconn_BA_count == 1)

In [7]:
@nb.njit(parallel=True, boundscheck=False, fastmath=True)
def count_bad_wspd_parallel(pos_in_tour, pairs, node_ranges, indices):
    #bad_count = 0
    num_pairs = len(pairs)
    is_bad = np.zeros(num_pairs, dtype=np.bool_)
    
    for i in nb.prange(num_pairs):
        a_node = pairs[i][0]
        b_node = pairs[i][1]

        a_start = node_ranges[a_node]['start']
        a_end = node_ranges[a_node]['end']
        b_start = node_ranges[b_node]['start']
        b_end = node_ranges[b_node]['end']
        
        # if both sets have more than 1 point, then we need to check the heuristic. Otherwise, it is automatically satisfied
        if (a_end - a_start) > 1 and (b_end - b_start) > 1:
            # 3. POSITION MAPPING & SORTING
            A = indices[a_start:a_end].copy()
            B = indices[b_start:b_end].copy()
            sort_by_key_inplace(A, pos_in_tour)
            sort_by_key_inplace(B, pos_in_tour)

            # 4. HEURISTIC CHECK
            if not wsp_heuristic_good(A, B, pos_in_tour):
                #bad_count += 1
                is_bad[i] = True

    return np.flatnonzero(is_bad)

def check_tour_with_wspd(wspd: tuple[np.ndarray, np.ndarray, np.ndarray], tour: np.ndarray):
    pairs, node_ranges, indices = wspd

    # vectorized node -> position map
    pos_in_tour = np.empty(len(tour), dtype=np.uint32)
    pos_in_tour[tour] = np.arange(len(tour), dtype=np.uint32)

    # 2. Execute parallel loop natively in Numba
    bad_count = count_bad_wspd_parallel(
        pos_in_tour,
        pairs,
        node_ranges,
        indices,
    )

    return bad_count

def check_problem_tour(prob: tsplib95.models.StandardProblem, tour: np.ndarray, s: float = S_FACTOR):
    """Given a TSP and a non-wrapping tour, checks if the WSP heuristic holds"""
    points = np.array([prob.node_coords[i] for i in prob.get_nodes()], dtype=np.float64)

    # Build the WSPDs with different separation factors
    pairs, node_ranges, indices = get_wspd(points, s, np.int64)
    print("built wspd1.5", flush=True)

    return check_tour_with_wspd((pairs, node_ranges, indices), tour), pairs, node_ranges, indices

In [8]:
#def profile_bottleneck(all_problems: list[tsplib95.models.StandardProblem]):

#for prob in all_problems:
#    print(f"Checking {prob.name}...", flush=True)

#    lkh_instance = elkai.Coordinates2D({
#        i-1: prob.node_coords[i] for i in prob.get_nodes()
#    })

#    # tour numpy conversions
#    tour_lkh = lkh_instance.solve_tsp(runs=1)
#    tour = np.array(tour_lkh, dtype=np.uint16)

#    check_problem_tour(prob, tour)

#%lprun -f profile_bottleneck -u 1.0 profile_bottleneck(all_problems)
#print("Done")

## Loading previously solved TSPs

In [9]:
mona_prob = tsplib95.load("ALL_tsp/mona-lisa100K.tsp")
mona_tour_prob = tsplib95.load("ALL_tsp/mona-lisa100K.tour")

In [10]:
mona_tour = np.array(mona_tour_prob.tours[0], dtype=np.uint32) - 1
mona_tour.shape

(100000,)

In [11]:
#from any_metric_wspd import gen_wspd
#from scipy.spatial.distance import pdist, squareform

#points = np.asarray([gaia_prob.node_coords[i] for i in gaia_prob.get_nodes()], dtype=np.float32)
#dist_matrix = pdist(points, metric="euclidean")
#dist_matrix = squareform(dist_matrix)
#dist_matrix.shape

#wspd = gen_wspd(dist_matrix, 0.9999)
#len(wspd)

In [12]:
#check_problem_tour(mona_prob, mona_tour)

#%lprun -f check_problem_tour -u 1.0 check_problem_tour(mona_prob, gaia_tour, dim=2)

In [13]:
## Scan for all "*.opt.tour" files and test matching EUC_2D / EUC_3D instances
#search_roots = [TSP_LIB_PATH, Path("ALL_tsp"), Path(".")]
#opt_tour_files = []

#seen = set()
#for root in search_roots:
#    if not root.exists():
#        continue
#    for p in root.rglob("*.opt.tour"):
#        rp = p.resolve()
#        if rp not in seen:
#            seen.add(rp)
#            opt_tour_files.append(p)

#opt_tour_files = sorted(opt_tour_files)

#results = []
#failures = []

#for opt_path in opt_tour_files:
#    try:
#        tsp_path = opt_path.with_suffix("").with_suffix(".tsp")  # *.opt.tour -> *.tsp
#        if not tsp_path.exists():
#            failures.append((str(opt_path), "matching .tsp not found"))
#            continue

#        prob = tsplib95.load(str(tsp_path))
#        if prob.edge_weight_type not in ("EUC_2D", "EUC_3D"):
#            continue

#        tour_prob = tsplib95.load(str(opt_path))
#        if not getattr(tour_prob, "tours", None):
#            failures.append((prob.name, "no tours in opt file"))
#            continue

#        dim = 2 if prob.edge_weight_type == "EUC_2D" else 3

#        # Robust node-id -> index mapping (does not assume nodes are exactly 1..n)
#        nodes = list(prob.get_nodes())
#        node_to_idx = {node_id: idx for idx, node_id in enumerate(nodes)}

#        raw_tour = tour_prob.tours[0]
#        try:
#            tour_idx = np.array([node_to_idx[n] for n in raw_tour], dtype=np.uint32)
#        except KeyError as e:
#            failures.append((prob.name, f"tour contains unknown node id: {e}"))
#            continue

#        # Ensure wrapped/closed tour for check_problem_tour()
#        if tour_idx[0] != tour_idx[-1]:
#            tour_idx = np.concatenate([tour_idx, tour_idx[:1]])

#        print(f"Checking {prob.name} ({prob.edge_weight_type}, n={prob.dimension})...", flush=True)
#        bad_count, biggest_pair = check_problem_tour(prob, tour_idx, dim=dim)

#        results.append({
#            "name": prob.name,
#            "type": prob.edge_weight_type,
#            "n": prob.dimension,
#            "bad_pairs": bad_count,
#            "biggest_pair": biggest_pair,
#            "opt_tour_file": str(opt_path),
#        })

#    except Exception as e:
#        failures.append((str(opt_path), repr(e)))

#print(f"\nChecked {len(results)} EUC_2D/EUC_3D problems with *.opt.tour")
#print(f"Failures: {len(failures)}")
#if failures:
#    print("First 10 failures:")
#    for item in failures[:10]:
#        print(" -", item)

## Quick aggregate summary
#if results:
#    total_bad = sum(r["bad_pairs"] for r in results)
#    bad_instances = sum(1 for r in results if r["bad_pairs"] > 0)
#    print(f"\nTotal bad pairs: {total_bad}")
#    print(f"Instances with >=1 bad pair: {bad_instances}/{len(results)}")

## Checking large instances

In [25]:
gaia_prob = tsplib95.load("ALL_tsp/gaia2079471.tsp") # gaia2079471
gaia_tour_prob = tsplib95.load("ALL_tsp/gaia2079471.tour")
gaia_tour = np.array(gaia_tour_prob.tours[0], dtype=np.uint32) - 1
pos_in_tour = np.empty(len(gaia_tour), dtype=np.uint32)
pos_in_tour[gaia_tour] = np.arange(len(gaia_tour), dtype=np.uint32)
points = np.array([gaia_prob.node_coords[i] for i in gaia_prob.get_nodes()], dtype=np.float64)

In [26]:
bad_pairs, pairs, node_ranges, indices = check_problem_tour(gaia_prob, gaia_tour, s=0.01)
len(bad_pairs)

built wspd1.5


25405

In [27]:
node_pairs = []

biggest_size = 0
for idx in bad_pairs:
    a_node, b_node = pairs[idx]

    a_start, a_end = node_ranges[a_node]
    b_start, b_end = node_ranges[b_node]

    A = indices[a_start:a_end]
    B = indices[b_start:b_end]

    print(f"Bad pair: {A} (size {len(A)}) and {B} (size {len(B)})")

    node_pairs.append((A, B))
    biggest_size = max(biggest_size, len(A) + len(B))
    
biggest_size

Bad pair: [1069790 1069946] (size 2) and [1069876 1064644 1064636 1064671 1064306 1064451 1064566 1064373 1064160
 1064122 1063918 1064030 1064144 1064159 1064132] (size 15)
Bad pair: [1831712 1831706] (size 2) and [1831737 1831764 1831575] (size 3)
Bad pair: [1073170 1093074 1076501] (size 3) and [1073390 1072849] (size 2)
Bad pair: [1093160 1093351 1093189 1093079 1093291 1093360] (size 6) and [1093322 1094193] (size 2)
Bad pair: [1095261 1096264 1096004] (size 3) and [1092505 1092572 1092554 1092171 1092487 1092529] (size 6)
Bad pair: [1091147 1091334 1091085] (size 3) and [1091089 1072468] (size 2)
Bad pair: [1158530 1158621 1158652] (size 3) and [1158714 1158218] (size 2)
Bad pair: [1833187 1091978 1091358 1091406 1092068 1092570 1091998 1092227 1832874
 1832855 1832523 1833042 1831623 1833117 1832914 1092328 1091147 1091334
 1091085 1831881 1831652 1831851 1831820 1831470 1831758 1091154 1832718
 1832697 1092026 1091984 1832836 1091388 1091558 1091565 1095261 1096264
 1096004 109

226

In [28]:
len(gaia_tour)

2079471

In [32]:
import utils.repair_euc, utils.helpers
importlib.reload(utils.repair_euc)
importlib.reload(utils.helpers)

from utils.repair_euc import repair_tour_euc
from utils.helpers import calc_tour_len_euc, valid_tour, roll_to_node

In [33]:
A, B = node_pairs[2947]
#A2, B2 = node_pairs[4033]
#A3, B3 = node_pairs[4559]
#A4, B4 = node_pairs[5451]
#A5, B5 = node_pairs[5602]
#A6, B6 = node_pairs[5912]
#AB = np.unique(np.concatenate((A4, A3, A6, B4, B3, B6, B[:1])))
original = gaia_tour.copy()
new_tour = repair_tour_euc(gaia_tour, A, B, points, HIGH=0)
valid_tour(new_tour, len(original))

True

In [34]:
original_len = calc_tour_len_euc(points, original)
original_len, calc_tour_len_euc(points, new_tour)

(288843524.0, 288843780.0)

In [35]:
#for i, (A, B) in enumerate(node_pairs):
#    original = gaia_tour.copy()
#    try:
#        new_tour = repair_tour_euc(gaia_tour, A, B, points)
#    except NotImplementedError as e:
#        #print(f"Error occurred while repairing tour for pair ({A}, {B}): {e}")
#        continue
#    assert valid_tour(new_tour, len(original)), "Repaired tour is not valid"
#    assert original_len == calc_tour_len_euc(points, new_tour), f"Repaired tour {i} has different length"

def test_repair_tour_euc(args):
    i, AB = args
    A, B = AB

    try:
        new_tour = repair_tour_euc(gaia_tour, A, B, points, 25)
    except NotImplementedError as e:
        #print(f"Error occurred while repairing tour for pair ({A}, {B}): {e}")
        return None
    assert valid_tour(new_tour, len(original)), "Repaired tour is not valid"
    if original_len > calc_tour_len_euc(points, new_tour):
        print(f"Repaired tour {i} has different length: original {original_len}, new {calc_tour_len_euc(points, new_tour)}")
    #assert original_len - 0.5 <= calc_tour_len_euc(points, new_tour), f"Repaired tour {i} has different length"
    #return calc_tour_len_euc(points, new_tour)

with ThreadPoolExecutor(max_workers=6) as executor:
    results = list(executor.map(test_repair_tour_euc, enumerate(node_pairs)))